In [1]:
import requests
import pandas as pd
import time

base_page = "https://www.kinometro.ru/kino/analitika/ystart/2015/yend/2026/country/6"

headers_page = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

headers_api = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*",
    "Referer": base_page,
    "X-Requested-With": "XMLHttpRequest",
}

session = requests.Session()

page_resp = session.get(base_page, headers=headers_page)
print("page status:", page_resp.status_code)

all_data = []

for page in range(96):
    start = page * 20

    api_url = (
        "https://www.kinometro.ru/kino/copytotal/"
        f"_dc/{int(time.time()*1000)}/"
        "sorting/st_date_start/order/DESC/"
        f"start/{start}/limit/20/"
        "filter/["
        "%7B%22property%22:%22ystart%22,%22value%22:2015%7D,"
        "%7B%22property%22:%22yend%22,%22value%22:2026%7D,"
        "%7B%22property%22:%22country%22,%22value%22:6%7D]"
    )

    r = session.get(api_url, headers=headers_api)
    data = r.json()

    print(f"page {page+1}: status={r.status_code}, rows={len(data['movie'])}")

    all_data.extend(data["movie"])

    time.sleep(0.5)

df = pd.DataFrame(all_data)

print(df.shape)
print(df.columns.tolist())
print(df.head())

df.to_csv("kinometro_films_raw.csv", index=False, encoding="utf-8-sig")

page status: 200
page 1: status=200, rows=20
page 2: status=200, rows=20
page 3: status=200, rows=20
page 4: status=200, rows=20
page 5: status=200, rows=20
page 6: status=200, rows=20
page 7: status=200, rows=20
page 8: status=200, rows=20
page 9: status=200, rows=20
page 10: status=200, rows=20
page 11: status=200, rows=20
page 12: status=200, rows=20
page 13: status=200, rows=20
page 14: status=200, rows=20
page 15: status=200, rows=20
page 16: status=200, rows=20
page 17: status=200, rows=20
page 18: status=200, rows=20
page 19: status=200, rows=20
page 20: status=200, rows=20
page 21: status=200, rows=20
page 22: status=200, rows=20
page 23: status=200, rows=20
page 24: status=200, rows=20
page 25: status=200, rows=20
page 26: status=200, rows=20
page 27: status=200, rows=20
page 28: status=200, rows=20
page 29: status=200, rows=20
page 30: status=200, rows=20
page 31: status=200, rows=20
page 32: status=200, rows=20
page 33: status=200, rows=20
page 34: status=200, rows=20
page 3

In [15]:
from bs4 import BeautifulSoup
import pandas as pd
import re

# копия исходного датафрейма
df_clean = df.copy()

# функция очистки текста
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("&nbsp;", "").replace("\xa0", " ")
    x = BeautifulSoup(x, "html.parser").get_text(" ", strip=True)
    x = re.sub(r"\s+", " ", x).strip()
    return x

# чистим все столбцы
for col in df_clean.columns:
    df_clean[col] = df_clean[col].apply(clean_text)

# дата в формат дд.мм.гг
df_clean["st_date_start"] = pd.to_datetime(df_clean["st_date_start"], errors="coerce").dt.strftime("%d.%m.%y")
df_clean["st_date_start"] = df_clean["st_date_start"].fillna("")

# собираем нужные 20 столбцов
df_clean = df_clean.rename(columns={
    "st_name_ru": "название",
    "st_distrib": "дистрибьютор",
    "st_date_start": "дата_старта",
    "st_total_copy": "тотал_экраны",
    "st_total_box": "тотал_сбор",
    "st_total_adm": "тотал_зритель",
    "st_total_avg": "тотал_наработка",
    "st_thursday_box": "сбор_чт",
    "st_thu_ratio": "множ_чт",
    "st_1wd_copy": "премьерный_уикенд_экраны",
    "st_1wd_box": "премьерный_уикенд_сбор",
    "st_1wd_adm": "премьерный_уикенд_зритель",
    "st_1wd_avg": "премьерный_уикенд_наработка",
    "st_2wd_box": "второй_уикенд_сбор",
    "st_2wd_adm": "второй_уикенд_зритель",
    "st_2wd_change": "второй_уикенд_падение",
    "st_prev_box": "превью_сбор",
    "st_prev_start": "превью_дни",
    "st_genre": "жанр",
    "st_country": "страна"
})

df_clean = df_clean[
    [
        "название",
        "дистрибьютор",
        "дата_старта",
        "тотал_экраны",
        "тотал_сбор",
        "тотал_зритель",
        "тотал_наработка",
        "сбор_чт",
        "множ_чт",
        "премьерный_уикенд_экраны",
        "премьерный_уикенд_сбор",
        "премьерный_уикенд_зритель",
        "премьерный_уикенд_наработка",
        "второй_уикенд_сбор",
        "второй_уикенд_зритель",
        "второй_уикенд_падение",
        "превью_сбор",
        "превью_дни",
        "жанр",
        "страна"
    ]
].copy()

# где после очистки осталась "0", можно заменить на пусто только в тех полях, где это выглядит как отсутствие данных
cols_optional = [
    "сбор_чт", "множ_чт", "второй_уикенд_сбор", "второй_уикенд_зритель",
    "второй_уикенд_падение", "превью_сбор", "превью_дни"
]
for col in cols_optional:
    df_clean[col] = df_clean[col].replace({"0": "", "946684800": ""})

# сохраняем
df_clean.to_csv("kinometro_films_clean_20cols.csv", index=False, encoding="utf-8-sig")

print(df_clean.head())
print(df_clean.shape)

                   название дистрибьютор дата_старта тотал_экраны тотал_сбор  \
0          ДОМОВЕНОК КУЗЯ 2           AK    19.03.26         2452    277 млн   
1                ДОКТОР ГАФ           CP    19.03.26         1579     25 млн   
2  КАРТИНЫ ДРУЖЕСКИХ СВЯЗЕЙ          PNR    19.03.26          116     10 млн   
3      МОИ ТАЕЖНЫЕ КАНИКУЛЫ         SMKT    19.03.26           83    5,6 млн   
4           КОЛОКОЛ НАДЕЖДЫ          KNT    12.03.26          189    754 тыс   

  тотал_зритель тотал_наработка сбор_чт множ_чт премьерный_уикенд_экраны  \
0       683 тыс         113 тыс                                     2452   
1        62 тыс          16 тыс                                     1579   
2        13 тыс          89 тыс                                      116   
3        13 тыс          67 тыс                                       83   
4       3,5 тыс         4,0 тыс                                      189   

  премьерный_уикенд_сбор премьерный_уикенд_зритель  \
0       